In [ ]:

from google.colab import drive
drive.mount("/content/drive")


In [ ]:

from pathlib import Path

project_root = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection"
)

print("PROJECT ROOT:", project_root)
print("EXISTS:", project_root.exists())
print("IS DIRECTORY:", project_root.is_dir())

if project_root.is_dir():
    all_items = sorted(
        project_root.rglob("*"),
        key=lambda p: str(p).lower()
    )

    files = [p for p in all_items if p.is_file()]
    folders = [p for p in all_items if p.is_dir()]

    print("\nFOLDERS:", len(folders))
    for folder in folders:
        print("DIR :", folder.relative_to(project_root))

    print("\nFILES:", len(files))
    for file in files:
        print(
            "FILE:",
            file.relative_to(project_root),
            "| Size:",
            file.stat().st_size,
            "bytes"
        )

    total_bytes = sum(f.stat().st_size for f in files)
    print("\nTOTAL FILE SIZE:", round(total_bytes / (1024**2), 2), "MB")
else:
    print("Project folder not found. Nothing has been changed.")


In [ ]:
from pathlib import Path
import zipfile

zip_path = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/Archive (2).zip"
)

print("ZIP exists:", zip_path.exists())

if zip_path.exists():
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        print("ZIP integrity:", zip_ref.testzip())
        print("Total items inside ZIP:", len(zip_ref.namelist()))
        print("\nFirst 20 items:")
        for item in zip_ref.namelist()[:20]:
            print(item)

In [ ]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
from collections import Counter

zip_path = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/Archive (2).zip"
)

with ZipFile(zip_path, "r") as zip_ref:
    files = [
        name for name in zip_ref.namelist()
        if not name.endswith("/")
    ]

    # Count files by folder
    folder_counts = Counter()

    # Count file extensions
    extension_counts = Counter()

    for name in files:
        path = PurePosixPath(name)

        if len(path.parts) >= 3:
            folder = "/".join(path.parts[:3])
        else:
            folder = "/".join(path.parts[:-1]) or "Root"

        folder_counts[folder] += 1
        extension_counts[path.suffix.lower()] += 1

    print("TOTAL FILES:", len(files))

    print("\nIMAGE COUNTS BY DATASET FOLDER:")
    for folder, count in sorted(folder_counts.items()):
        print(f"{folder}: {count}")

    print("\nFILE TYPES:")
    for extension, count in extension_counts.items():
        print(f"{extension or 'No extension'}: {count}")

In [ ]:
from pathlib import Path
from zipfile import ZipFile

zip_path = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/Archive (2).zip"
)

extract_path = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/extracted"
)

# Prevent accidental overwrite
if extract_path.exists() and any(extract_path.iterdir()):
    print("Extraction folder already contains files.")
    print("No extraction performed.")
else:
    extract_path.mkdir(parents=True, exist_ok=True)

    with ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

    print("Dataset extracted successfully!")

print("\nExtracted folders:")
for folder in sorted(extract_path.rglob("*")):
    if folder.is_dir():
        print(folder.relative_to(extract_path))

In [ ]:
from pathlib import Path
from PIL import Image
from collections import defaultdict

data_path = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/extracted/Data"
)

folders = {
    "Train Fire": data_path / "Train_Data" / "Fire",
    "Train Non-Fire": data_path / "Train_Data" / "Non_Fire",
    "Test Fire": data_path / "Test_Data" / "Fire",
    "Test Non-Fire": data_path / "Test_Data" / "Non_Fire",
}

total_images = 0
total_corrupted = 0

print("IMAGE VERIFICATION REPORT\n")

for name, folder in folders.items():
    images = list(folder.glob("*.jpg"))
    valid_count = 0
    corrupted = []

    for image_path in images:
        try:
            with Image.open(image_path) as img:
                img.verify()
            valid_count += 1
        except Exception:
            corrupted.append(image_path.name)

    total_images += len(images)
    total_corrupted += len(corrupted)

    print(f"{name}:")
    print(f"  Total images: {len(images)}")
    print(f"  Valid images: {valid_count}")
    print(f"  Corrupted images: {len(corrupted)}")

    if corrupted:
        print("  Corrupted filenames:", corrupted[:10])

    print()

print("=" * 40)
print("TOTAL IMAGES:", total_images)
print("TOTAL CORRUPTED:", total_corrupted)
print("TOTAL VALID:", total_images - total_corrupted)

In [ ]:
from pathlib import Path
from PIL import Image
from collections import Counter

fire_path = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/extracted/Data/Train_Data/Fire"
)

corrupted_files = []

for image_path in fire_path.glob("*.jpg"):
    try:
        with Image.open(image_path) as img:
            img.verify()
    except Exception as error:
        corrupted_files.append((image_path, str(error)))

print("TOTAL CORRUPTED:", len(corrupted_files))

print("\nERROR SUMMARY:")
error_counts = Counter(error for _, error in corrupted_files)

for error, count in error_counts.most_common():
    print(f"{count} files: {error[:200]}")

print("\nFIRST 20 CORRUPTED FILES:")
for image_path, error in corrupted_files[:20]:
    print(
        f"{image_path.name} | "
        f"Size: {image_path.stat().st_size} bytes | "
        f"Error: {error[:150]}"
    )

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from collections import Counter

zip_path = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/Archive (2).zip"
)

extract_path = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/extracted"
)

fire_path = extract_path / "Data/Train_Data/Fire"

zero_byte_files = [
    p for p in fire_path.glob("*.jpg")
    if p.stat().st_size == 0
]

print("ZERO-BYTE FILES FOUND:", len(zero_byte_files))

zip_zero_byte = []
zip_nonzero = []
zip_missing = []

with ZipFile(zip_path, "r") as zip_ref:
    for file_path in zero_byte_files:
        zip_name = "Data/Train_Data/Fire/" + file_path.name

        try:
            info = zip_ref.getinfo(zip_name)

            if info.file_size == 0:
                zip_zero_byte.append(file_path.name)
            else:
                zip_nonzero.append((file_path.name, info.file_size))

        except KeyError:
            zip_missing.append(file_path.name)

print("\nORIGINAL ZIP INSPECTION")
print("Zero-byte entries inside ZIP:", len(zip_zero_byte))
print("Non-zero entries inside ZIP:", len(zip_nonzero))
print("Missing entries inside ZIP:", len(zip_missing))

print("\nFIRST 15 NON-ZERO ZIP ENTRIES:")
for name, size in zip_nonzero[:15]:
    print(name, "|", size, "bytes")

print("\nFIRST 15 ZERO-BYTE ZIP ENTRIES:")
for name in zip_zero_byte[:15]:
    print(name)

In [ ]:
from pathlib import Path

dataset_path = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/extracted/Data"
)

folders = [
    "Train_Data/Fire",
    "Train_Data/Non_Fire",
    "Test_Data/Fire",
    "Test_Data/Non_Fire"
]

print("ZERO-BYTE FILE CHECK\n")

for folder in folders:
    folder_path = dataset_path / folder

    zero_files = [
        p.name for p in folder_path.glob("*.jpg")
        if p.stat().st_size == 0
    ]

    print(f"{folder}: {len(zero_files)} zero-byte files")

In [ ]:
from pathlib import Path
import shutil

source = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/extracted/Data"
)

destination = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/clean"
)

folders = [
    "Train_Data/Fire",
    "Train_Data/Non_Fire",
    "Test_Data/Fire",
    "Test_Data/Non_Fire"
]

for folder in folders:
    source_folder = source / folder
    destination_folder = destination / folder

    destination_folder.mkdir(parents=True, exist_ok=True)

    copied = 0
    skipped = 0

    for image in source_folder.glob("*.jpg"):

        # Skip empty files
        if image.stat().st_size == 0:
            skipped += 1
            continue

        shutil.copy2(image, destination_folder / image.name)
        copied += 1

    print(f"\n{folder}")
    print("Valid images copied:", copied)
    print("Empty images skipped:", skipped)

print("\nClean dataset created successfully!")

In [ ]:
from pathlib import Path

clean_path = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/clean"
)

folders = [
    "Train_Data/Fire",
    "Train_Data/Non_Fire",
    "Test_Data/Fire",
    "Test_Data/Non_Fire"
]

total_images = 0
total_empty = 0

print("CLEAN DATASET VERIFICATION\n")

for folder in folders:
    folder_path = clean_path / folder

    images = list(folder_path.glob("*.jpg"))
    empty_images = [
        image for image in images
        if image.stat().st_size == 0
    ]

    total_images += len(images)
    total_empty += len(empty_images)

    print(f"{folder}:")
    print("Total images:", len(images))
    print("Zero-byte images:", len(empty_images))
    print()

print("FINAL SUMMARY")
print("Total images:", total_images)
print("Total zero-byte images:", total_empty)

In [ ]:
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split

# Source dataset
source = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/clean"
)

# Destination dataset
destination = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/prepared"
)

classes = ["Fire", "Non_Fire"]

# Create train and validation folders
for split in ["train", "val", "test"]:
    for class_name in classes:
        (destination / split / class_name).mkdir(
            parents=True,
            exist_ok=True
        )

# Split training images class-wise
for class_name in classes:

    image_files = sorted(
        (source / "Train_Data" / class_name).glob("*.jpg")
    )

    train_files, val_files = train_test_split(
        image_files,
        test_size=0.10,
        random_state=42,
        shuffle=True
    )

    # Copy training images
    for image in train_files:
        shutil.copy2(
            image,
            destination / "train" / class_name / image.name
        )

    # Copy validation images
    for image in val_files:
        shutil.copy2(
            image,
            destination / "val" / class_name / image.name
        )

    print(f"\n{class_name}")
    print("Training:", len(train_files))
    print("Validation:", len(val_files))

# Copy original test images without splitting
for class_name in classes:

    test_files = (
        source / "Test_Data" / class_name
    ).glob("*.jpg")

    copied = 0

    for image in test_files:
        shutil.copy2(
            image,
            destination / "test" / class_name / image.name
        )
        copied += 1

    print(f"\nTest {class_name}: {copied}")

print("\nDataset preparation completed!")

In [ ]:
from pathlib import Path

prepared_path = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/prepared"
)

splits = ["train", "val", "test"]
classes = ["Fire", "Non_Fire"]

total_images = 0

for split in splits:
    print(f"\n{split.upper()} DATASET")

    for class_name in classes:
        folder = prepared_path / split / class_name

        images = list(folder.glob("*.jpg"))

        zero_files = [
            img for img in images
            if img.stat().st_size == 0
        ]

        print(
            f"{class_name}: {len(images)} images | "
            f"Zero-byte files: {len(zero_files)}"
        )

        total_images += len(images)

print("\nTOTAL PREPARED IMAGES:", total_images)

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

dataset_path = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/prepared/train"
)

classes = ["Fire", "Non_Fire"]

plt.figure(figsize=(14, 8))

for row, class_name in enumerate(classes):

    image_folder = dataset_path / class_name
    image_files = list(image_folder.glob("*.jpg"))

    selected_images = random.sample(image_files, 4)

    for col, image_path in enumerate(selected_images):

        image = Image.open(image_path).convert("RGB")

        plt.subplot(2, 4, row * 4 + col + 1)
        plt.imshow(image)
        plt.title(class_name)
        plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available. Training will use CPU.")

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

DATASET_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/prepared"
)

for split in ["train", "val", "test"]:
    print(f"\n{split.upper()}")

    for class_name in ["Fire", "Non_Fire"]:
        folder = DATASET_PATH / split / class_name
        count = len(list(folder.glob("*.jpg")))

        print(f"{class_name}: {count}")

In [ ]:
from pathlib import Path

drive_path = Path("/content/drive/MyDrive")
project_path = drive_path / "WildForest_Fire_Detection"

print("Google Drive mounted:", drive_path.exists())
print("MyDrive accessible:", drive_path.is_dir())

print("\nMyDrive folders:")
if drive_path.exists():
    for item in drive_path.iterdir():
        print(item.name)

print("\nProject folder exists:", project_path.exists())

if project_path.exists():
    print("\nProject contents:")
    for item in project_path.iterdir():
        print(item.name)

In [ ]:
from pathlib import Path

base = Path("/content/drive/MyDrive")

for project_name in [
    "WildForest_Fire_Detection",
    "WildForest_Fire_Detection (1)"
]:
    project = base / project_name

    print(f"\n========== {project_name} ==========")

    if not project.exists():
        print("Folder not found")
        continue

    for path in project.rglob("*"):
        if path.is_dir():
            print("FOLDER:", path)

In [ ]:
from pathlib import Path

drive = Path("/content/drive/MyDrive")

print("SEARCHING FOR PREPARED DATASET...\n")

for path in drive.rglob("prepared"):
    if path.is_dir():
        print("FOUND:", path)

        for split in ["train", "val", "test"]:
            split_path = path / split

            if split_path.exists():
                print(
                    f"  {split}:",
                    sum(1 for _ in split_path.rglob("*.jpg")),
                    "images"
                )

In [ ]:
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split

source = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/clean"
)

destination = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/prepared"
)

classes = ["Fire", "Non_Fire"]

# Create the destination folders
for split in ["train", "val", "test"]:
    for class_name in classes:
        (destination / split / class_name).mkdir(
            parents=True,
            exist_ok=True
        )

# Create training and validation splits
for class_name in classes:

    images = sorted(
        (source / "Train_Data" / class_name).glob("*.jpg")
    )

    train_images, val_images = train_test_split(
        images,
        test_size=0.10,
        random_state=42,
        shuffle=True
    )

    for image in train_images:
        shutil.copy2(
            image,
            destination / "train" / class_name / image.name
        )

    for image in val_images:
        shutil.copy2(
            image,
            destination / "val" / class_name / image.name
        )

    print(f"\n{class_name}")
    print("Training:", len(train_images))
    print("Validation:", len(val_images))

# Copy the original test images
for class_name in classes:

    images = (
        source / "Test_Data" / class_name
    ).glob("*.jpg")

    for image in images:
        shutil.copy2(
            image,
            destination / "test" / class_name / image.name
        )

    print(f"Test {class_name}: {len(list((source / 'Test_Data' / class_name).glob('*.jpg')))}")

print("\nPrepared dataset recreated successfully!")
print("Location:", destination)

In [ ]:
from pathlib import Path

base = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset"
)

folders = {
    "Clean Train Fire": base / "clean/Train_Data/Fire",
    "Clean Train Non-Fire": base / "clean/Train_Data/Non_Fire",
    "Clean Test Fire": base / "clean/Test_Data/Fire",
    "Clean Test Non-Fire": base / "clean/Test_Data/Non_Fire",
    "Prepared Train Fire": base / "prepared/train/Fire",
    "Prepared Train Non-Fire": base / "prepared/train/Non_Fire",
    "Prepared Val Fire": base / "prepared/val/Fire",
    "Prepared Val Non-Fire": base / "prepared/val/Non_Fire",
    "Prepared Test Fire": base / "prepared/test/Fire",
    "Prepared Test Non-Fire": base / "prepared/test/Non_Fire",
}

for name, folder in folders.items():

    if folder.exists():
        images = list(folder.glob("*.jpg"))
        print(f"{name}: {len(images)} images")
    else:
        print(f"{name}: FOLDER MISSING")

In [ ]:
from pathlib import Path

base = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/extracted/Data"
)

folders = [
    "Train_Data/Fire",
    "Train_Data/Non_Fire",
    "Test_Data/Fire",
    "Test_Data/Non_Fire"
]

print("ORIGINAL EXTRACTED DATASET\n")

for folder in folders:
    path = base / folder

    images = list(path.glob("*.jpg"))

    valid_images = [
        image for image in images
        if image.stat().st_size > 0
    ]

    print(f"{folder}")
    print("Total JPG files:", len(images))
    print("Valid images:", len(valid_images))
    print("Empty images:", len(images) - len(valid_images))
    print()

In [ ]:
from pathlib import Path
import shutil

source = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/extracted/Data"
)

destination = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/clean"
)

folders = [
    "Train_Data/Fire",
    "Train_Data/Non_Fire",
    "Test_Data/Fire",
    "Test_Data/Non_Fire"
]

for folder in folders:

    source_folder = source / folder
    destination_folder = destination / folder

    # Create destination folder if missing
    destination_folder.mkdir(parents=True, exist_ok=True)

    copied = 0
    skipped = 0

    for image in source_folder.glob("*.jpg"):

        if image.stat().st_size == 0:
            skipped += 1
            continue

        shutil.copy2(
            image,
            destination_folder / image.name
        )

        copied += 1

    print(f"\n{folder}")
    print("Valid images copied:", copied)
    print("Empty images skipped:", skipped)

print("\nCLEAN DATASET REBUILT SUCCESSFULLY!")

In [ ]:
from pathlib import Path

clean_path = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/clean"
)

folders = [
    "Train_Data/Fire",
    "Train_Data/Non_Fire",
    "Test_Data/Fire",
    "Test_Data/Non_Fire"
]

total = 0

for folder in folders:
    path = clean_path / folder

    images = list(path.glob("*.jpg"))
    empty = [img for img in images if img.stat().st_size == 0]

    print(f"{folder}: {len(images)} images")
    print(f"Zero-byte files: {len(empty)}\n")

    total += len(images)

print("TOTAL CLEAN IMAGES:", total)

In [ ]:
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split

source = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/clean"
)

destination = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/prepared"
)

classes = ["Fire", "Non_Fire"]

# Remove only the previous incomplete prepared dataset
if destination.exists():
    shutil.rmtree(destination)

print("Previous prepared folder cleared.")

# Create fresh folders
for split in ["train", "val", "test"]:
    for class_name in classes:
        (destination / split / class_name).mkdir(
            parents=True,
            exist_ok=True
        )

# Create training and validation sets
for class_name in classes:

    images = sorted(
        (source / "Train_Data" / class_name).glob("*.jpg")
    )

    train_images, val_images = train_test_split(
        images,
        test_size=0.10,
        random_state=42,
        shuffle=True
    )

    for image in train_images:
        shutil.copy2(
            image,
            destination / "train" / class_name / image.name
        )

    for image in val_images:
        shutil.copy2(
            image,
            destination / "val" / class_name / image.name
        )

    print(f"\n{class_name}")
    print("Training:", len(train_images))
    print("Validation:", len(val_images))

# Copy original test images
for class_name in classes:

    images = list(
        (source / "Test_Data" / class_name).glob("*.jpg")
    )

    for image in images:
        shutil.copy2(
            image,
            destination / "test" / class_name / image.name
        )

    print(f"Test {class_name}: {len(images)}")

# Final automatic verification
expected = {
    ("train", "Fire"): 1899,
    ("train", "Non_Fire"): 2250,
    ("val", "Fire"): 212,
    ("val", "Non_Fire"): 250,
    ("test", "Fire"): 25,
    ("test", "Non_Fire"): 25
}

print("\nFINAL VERIFICATION")

all_correct = True

for (split, class_name), expected_count in expected.items():

    folder = destination / split / class_name
    actual_count = len(list(folder.glob("*.jpg")))

    status = "PASS" if actual_count == expected_count else "FAIL"

    print(
        f"{split}/{class_name}: "
        f"{actual_count} | Expected: {expected_count} | {status}"
    )

    if actual_count != expected_count:
        all_correct = False

print("\nRESULT:", "ALL CHECKS PASSED" if all_correct else "CHECK FAILED")

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms

from torchvision import datasets
from torch.utils.data import DataLoader

from pathlib import Path

# Dataset location
DATASET_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/prepared"
)

# GPU configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
train_dataset = datasets.ImageFolder(
    DATASET_PATH / "train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    DATASET_PATH / "val",
    transform=test_transform
)

test_dataset = datasets.ImageFolder(
    DATASET_PATH / "test",
    transform=test_transform
)

print("Class mapping:", train_dataset.class_to_idx)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Testing images:", len(test_dataset))

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders created successfully!")

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

In [ ]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

print("Image data type:", images.dtype)
print("Label data type:", labels.dtype)

print("First 10 labels:", labels[:10].tolist())
print("Class mapping:", train_dataset.class_to_idx)

In [ ]:
import torch.nn as nn
from torchvision import models

# Load ResNet-18 with pretrained ImageNet weights
weights = models.ResNet18_Weights.DEFAULT

model = models.resnet18(weights=weights)

# Get the number of input features in the final layer
num_features = model.fc.in_features

# Replace the final layer for binary classification
model.fc = nn.Linear(num_features, 2)

# Move model to GPU
model = model.to(device)

print("ResNet-18 loaded successfully!")
print("Input features:", num_features)
print("Output classes:", model.fc.out_features)
print("Model device:", next(model.parameters()).device)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Calculate class weights
class_counts = torch.tensor(
    [1899, 2250],
    dtype=torch.float32
)

class_weights = class_counts.sum() / (2 * class_counts)

class_weights = class_weights.to(device)

# Loss function
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

print("Loss function:", criterion)
print("Class weights:", class_weights)
print("Optimizer:", optimizer)
print("Learning rate:", optimizer.param_groups[0]["lr"])

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward propagation
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        # Calculate predictions
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = 100 * correct / total

    return epoch_loss, epoch_accuracy


def evaluate_model(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = 100 * correct / total

    return epoch_loss, epoch_accuracy

print("Training and evaluation functions defined successfully!")

In [ ]:
import copy
import torch

EPOCHS = 15

best_val_accuracy = 0.0
best_model_weights = copy.deepcopy(model.state_dict())

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

best_model_path = (
    "/content/drive/MyDrive/WildForest_Fire_Detection/weights/"
    "best_resnet18.pth"
)

from pathlib import Path
Path(best_model_path).parent.mkdir(parents=True, exist_ok=True)

for epoch in range(EPOCHS):

    train_loss, train_accuracy = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    val_loss, val_accuracy = evaluate_model(
        model, val_loader, criterion, device
    )

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)

    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    print(f"\nEpoch [{epoch + 1}/{EPOCHS}]")

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_accuracy:.2f}%")

    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.2f}%")

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        best_model_weights = copy.deepcopy(model.state_dict())

        torch.save(
            best_model_weights,
            best_model_path
        )

        print("Best model saved!")

print("\nTRAINING COMPLETED!")
print("Best Validation Accuracy:", round(best_val_accuracy, 2), "%")
print("Best model location:", best_model_path)

In [ ]:

# STEP 13: Load the best saved model

import torch
from pathlib import Path

MODEL_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/weights/best_resnet18.pth"
)

# Load the best checkpoint
model.load_state_dict(
    torch.load(MODEL_PATH, map_location=device, weights_only=True)
)

model = model.to(device)
model.eval()

print("Best ResNet-18 model loaded successfully!")
print("Model device:", next(model.parameters()).device)
print("Model is in evaluation mode:", not model.training)


In [ ]:

# STEP 14: Evaluate model on test dataset

import torch

model.eval()

test_correct = 0
test_total = 0
test_loss = 0.0

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = 100 * test_correct / test_total
average_test_loss = test_loss / test_total

print("TEST RESULTS")
print("-" * 35)
print(f"Test Images: {test_total}")
print(f"Correct Predictions: {test_correct}")
print(f"Incorrect Predictions: {test_total - test_correct}")
print(f"Test Loss: {average_test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")


In [ ]:

# STEP 15: Classification Report and Confusion Matrix

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# Class names in the correct label order
class_names = ["Fire", "Non_Fire"]

# Classification report
print("CLASSIFICATION REPORT")
print("-" * 50)

print(classification_report(
    all_labels,
    all_predictions,
    target_names=class_names,
    digits=4,
    zero_division=0
))

# Confusion matrix
cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("CONFUSION MATRIX")
print(cm)

# Display confusion matrix
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

display.plot(cmap="Blues", values_format="d")
plt.title("ResNet-18 - Fire vs Non-Fire")
plt.show()


In [ ]:

# STEP 16: Find incorrectly classified test images

wrong_predictions = []

model.eval()

with torch.no_grad():
    for index in range(len(test_dataset)):
        image, actual_label = test_dataset[index]

        input_image = image.unsqueeze(0).to(device)

        output = model(input_image)
        probabilities = torch.softmax(output, dim=1)

        predicted_label = torch.argmax(probabilities, dim=1).item()

        if predicted_label != actual_label:
            wrong_predictions.append({
                "index": index,
                "actual": test_dataset.classes[actual_label],
                "predicted": test_dataset.classes[predicted_label],
                "confidence": probabilities[0][predicted_label].item() * 100
            })

print("INCORRECT PREDICTIONS")
print("-" * 40)

for item in wrong_predictions:
    print("Image index:", item["index"])
    print("Actual class:", item["actual"])
    print("Predicted class:", item["predicted"])
    print(f"Confidence: {item['confidence']:.2f}%")


In [ ]:

# STEP 17: Visualize the incorrectly classified image

import matplotlib.pyplot as plt
import torch

# Get the incorrect image
wrong_index = wrong_predictions[0]["index"]

image, actual_label = test_dataset[wrong_index]

# Convert normalized tensor back to displayable image
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

display_image = image * std + mean
display_image = display_image.clamp(0, 1)

plt.figure(figsize=(7, 6))
plt.imshow(display_image.permute(1, 2, 0))
plt.axis("off")

plt.title(
    f"Actual: {test_dataset.classes[actual_label]}\n"
    f"Predicted: Fire (88.37% confidence)"
)

plt.show()


In [ ]:

# STEP 18: Save current model evaluation results

import os
import json
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix

# Create results directory
RESULTS_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/results/resnet18_baseline"
)

RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# 1. Save test metrics
metrics = {
    "model": "ResNet-18",
    "task": "Fire vs Non-Fire Image Classification",
    "test_images": int(test_total),
    "correct_predictions": int(test_correct),
    "incorrect_predictions": int(test_total - test_correct),
    "test_accuracy": float(test_accuracy),
    "test_loss": float(average_test_loss),
    "best_validation_accuracy": 99.78,
    "best_epoch": 11
}

with open(RESULTS_PATH / "test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

# 2. Save classification report
report = classification_report(
    all_labels,
    all_predictions,
    target_names=["Fire", "Non_Fire"],
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report).transpose().to_csv(
    RESULTS_PATH / "classification_report.csv"
)

# 3. Save confusion matrix
cm = confusion_matrix(all_labels, all_predictions)

pd.DataFrame(
    cm,
    index=["Actual_Fire", "Actual_Non_Fire"],
    columns=["Predicted_Fire", "Predicted_Non_Fire"]
).to_csv(RESULTS_PATH / "confusion_matrix.csv")

# Save confusion matrix visualization
from sklearn.metrics import ConfusionMatrixDisplay

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Fire", "Non_Fire"]
)

disp.plot(cmap="Blues", values_format="d")
plt.title("ResNet-18 Baseline Confusion Matrix")
plt.savefig(
    RESULTS_PATH / "confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()

# 4. Save incorrect predictions
pd.DataFrame(wrong_predictions).to_csv(
    RESULTS_PATH / "incorrect_predictions.csv",
    index=False
)

# Display saved files
print("\nBASELINE RESULTS SAVED SUCCESSFULLY!")
print("-" * 45)

for file in sorted(RESULTS_PATH.iterdir()):
    print(file.name)

print("\nLocation:", RESULTS_PATH)


In [ ]:

# STEP 19: Automated hard-negative image collection

import requests
import os
import hashlib
import time
import csv
from pathlib import Path

# Separate folder for new images
COLLECTION_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/hard_negatives/raw"
)

COLLECTION_PATH.mkdir(parents=True, exist_ok=True)

# Search categories: all are intended to find NON-FIRE scenes
SEARCH_TERMS = [
    "sunrise forest",
    "sunset forest",
    "sunlight through trees",
    "orange sky forest",
    "autumn forest",
    "foggy forest"
]

MAX_IMAGES_PER_CATEGORY = 20

API_URL = "https://commons.wikimedia.org/w/api.php"

session = requests.Session()
session.headers.update({
    "User-Agent": "WildForestFireDetection/1.0 (educational dataset project)"
})

# Only accept images with these reuse-friendly licenses
ALLOWED_LICENSES = [
    "CC0",
    "Public domain",
    "CC BY",
    "CC BY-SA"
]

downloaded_hashes = set()
metadata = []
download_count = 0

for search_term in SEARCH_TERMS:

    print(f"\nSearching: {search_term}")

    params = {
        "action": "query",
        "generator": "search",
        "gsrsearch": search_term + " filetype:bitmap",
        "gsrnamespace": 6,
        "gsrlimit": MAX_IMAGES_PER_CATEGORY,
        "prop": "imageinfo",
        "iiprop": "url|extmetadata",
        "iiurlwidth": 1024,
        "format": "json"
    }

    try:
        response = session.get(
            API_URL,
            params=params,
            timeout=30
        )
        response.raise_for_status()

        pages = response.json().get("query", {}).get("pages", {})

        category_count = 0

        for page in pages.values():

            image_info = page.get("imageinfo", [{}])[0]

            image_url = image_info.get("thumburl") or image_info.get("url")
            extmetadata = image_info.get("extmetadata", {})

            license_name = extmetadata.get(
                "LicenseShortName", {}
            ).get("value", "")

            # Skip images without an accepted license
            if not any(
                license_name.startswith(license)
                for license in ALLOWED_LICENSES
            ):
                continue

            if not image_url:
                continue

            try:
                image_response = session.get(
                    image_url,
                    timeout=30
                )
                image_response.raise_for_status()

                image_bytes = image_response.content

                # Basic file validation
                if len(image_bytes) < 5000:
                    continue

                # Avoid duplicate downloads
                image_hash = hashlib.sha256(
                    image_bytes
                ).hexdigest()

                if image_hash in downloaded_hashes:
                    continue

                downloaded_hashes.add(image_hash)

                safe_name = (
                    search_term.replace(" ", "_")
                    + "_"
                    + str(download_count + 1)
                    + ".jpg"
                )

                file_path = COLLECTION_PATH / safe_name

                with open(file_path, "wb") as f:
                    f.write(image_bytes)

                metadata.append([
                    safe_name,
                    search_term,
                    license_name,
                    image_url,
                    page.get("title", "")
                ])

                download_count += 1
                category_count += 1

                print("Downloaded:", safe_name)

                time.sleep(0.3)

            except Exception as e:
                print("Skipped image:", str(e)[:100])

        print("Category downloads:", category_count)

    except Exception as e:
        print("Search failed:", str(e)[:200])

# Save metadata for license tracking
metadata_file = COLLECTION_PATH / "image_metadata.csv"

with open(metadata_file, "w", newline="", encoding="utf-8") as f:

    writer = csv.writer(f)

    writer.writerow([
        "filename",
        "search_category",
        "license",
        "source_url",
        "commons_page"
    ])

    writer.writerows(metadata)

print("\n" + "=" * 45)
print("IMAGE COLLECTION COMPLETED")
print("=" * 45)
print("Total downloaded:", download_count)
print("Collection folder:", COLLECTION_PATH)
print("Metadata file:", metadata_file)


In [ ]:

# STEP 20: Validate downloaded images

from PIL import Image, UnidentifiedImageError
from pathlib import Path
import hashlib
import pandas as pd

COLLECTION_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/hard_negatives/raw"
)

valid_images = []
invalid_images = []
seen_hashes = set()
duplicate_images = []

for file_path in COLLECTION_PATH.iterdir():

    if file_path.name == "image_metadata.csv":
        continue

    if not file_path.is_file():
        continue

    try:
        # Verify image integrity
        with Image.open(file_path) as img:
            img.verify()

        # Read actual image properties
        with Image.open(file_path) as img:
            width, height = img.size
            image_format = img.format

        if width < 100 or height < 100:
            invalid_images.append(file_path.name)
            continue

        # Check exact duplicate files
        file_hash = hashlib.sha256(
            file_path.read_bytes()
        ).hexdigest()

        if file_hash in seen_hashes:
            duplicate_images.append(file_path.name)
            continue

        seen_hashes.add(file_hash)

        valid_images.append({
            "filename": file_path.name,
            "width": width,
            "height": height,
            "format": image_format
        })

    except Exception:
        invalid_images.append(file_path.name)

# Save validation report
validation_df = pd.DataFrame(valid_images)

validation_df.to_csv(
    COLLECTION_PATH / "validation_report.csv",
    index=False
)

print("IMAGE VALIDATION RESULTS")
print("-" * 40)
print("Valid images:", len(valid_images))
print("Invalid images:", len(invalid_images))
print("Exact duplicates:", len(duplicate_images))

print("\nImage formats:")
if not validation_df.empty:
    print(validation_df["format"].value_counts())

print("\nValidation report saved successfully!")


In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import math

COLLECTION_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/hard_negatives/raw"
)

image_files = sorted([
    p for p in COLLECTION_PATH.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]
])

BATCH_SIZE = 20

for start in range(0, len(image_files), BATCH_SIZE):

    batch = image_files[start:start + BATCH_SIZE]

    rows = math.ceil(len(batch) / 4)

    fig, axes = plt.subplots(rows, 4, figsize=(16, rows * 3))

    axes = axes.flatten()

    for ax, image_path in zip(axes, batch):
        with Image.open(image_path) as img:
            ax.imshow(img.convert("RGB"))

        ax.set_title(image_path.name, fontsize=8)
        ax.axis("off")

    for ax in axes[len(batch):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
from pathlib import Path
from PIL import Image
import shutil

RAW_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/hard_negatives/raw"
)

APPROVED_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/hard_negatives/approved/Non_Fire"
)

APPROVED_PATH.mkdir(parents=True, exist_ok=True)

image_files = sorted([
    p for p in RAW_PATH.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]
])

approved_count = 0

for image_path in image_files:

    try:
        with Image.open(image_path) as img:
            img = img.convert("RGB")

            output_path = APPROVED_PATH / f"{image_path.stem}.jpg"
            img.save(output_path, "JPEG", quality=95)

            approved_count += 1

    except Exception as e:
        print("Could not process:", image_path.name, e)

print("APPROVED DATASET CREATED")
print("-" * 40)
print("Images approved:", approved_count)
print("Approved folder:", APPROVED_PATH)
print("Total files in approved folder:", len(list(APPROVED_PATH.glob("*.jpg"))))

In [ ]:
from pathlib import Path
import shutil

BASE_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset"
)

ORIGINAL_PATH = BASE_PATH / "prepared"
NEW_PATH = BASE_PATH / "prepared_v2"

HARD_NEGATIVE_PATH = (
    BASE_PATH / "hard_negatives/approved/Non_Fire"
)

# Safety check: never overwrite an existing dataset version.
if NEW_PATH.exists():
    raise FileExistsError(
        f"{NEW_PATH} already exists. We will not overwrite it."
    )

# Copy the original prepared dataset.
shutil.copytree(ORIGINAL_PATH, NEW_PATH)

# Add hard-negative images only to the training Non_Fire folder.
TARGET_PATH = NEW_PATH / "train" / "Non_Fire"

added_count = 0

for image_path in HARD_NEGATIVE_PATH.glob("*.jpg"):
    destination = TARGET_PATH / f"hard_negative_{image_path.name}"
    shutil.copy2(image_path, destination)
    added_count += 1

print("NEW DATASET CREATED")
print("-" * 40)
print("Hard-negative images added:", added_count)

for split in ["train", "val", "test"]:
    print(f"\n{split.upper()}")

    for class_name in ["Fire", "Non_Fire"]:
        folder = NEW_PATH / split / class_name
        count = len(list(folder.glob("*")))
        print(f"{class_name}: {count}")

print("\nNew dataset location:", NEW_PATH)
print("Original dataset preserved:", ORIGINAL_PATH.exists())

In [ ]:
from pathlib import Path
from torchvision import datasets
from torch.utils.data import DataLoader

DATASET_V2_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/dataset/prepared_v2"
)

# Load datasets
train_dataset_v2 = datasets.ImageFolder(
    DATASET_V2_PATH / "train",
    transform=train_transform
)

val_dataset_v2 = datasets.ImageFolder(
    DATASET_V2_PATH / "val",
    transform=test_transform
)

test_dataset_v2 = datasets.ImageFolder(
    DATASET_V2_PATH / "test",
    transform=test_transform
)

# Loaders
train_loader_v2 = DataLoader(
    train_dataset_v2,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader_v2 = DataLoader(
    val_dataset_v2,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

test_loader_v2 = DataLoader(
    test_dataset_v2,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print("DATASET V2 LOADED SUCCESSFULLY")
print("-" * 40)
print("Class mapping:", train_dataset_v2.class_to_idx)
print("Training images:", len(train_dataset_v2))
print("Validation images:", len(val_dataset_v2))
print("Testing images:", len(test_dataset_v2))
print("Training batches:", len(train_loader_v2))
print("Validation batches:", len(val_loader_v2))
print("Testing batches:", len(test_loader_v2))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# Reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Load pretrained ResNet-18
weights = models.ResNet18_Weights.DEFAULT
model_v2 = models.resnet18(weights=weights)

# Replace final layer for 2 classes
num_features = model_v2.fc.in_features
model_v2.fc = nn.Linear(num_features, 2)

# Move model to GPU
model_v2 = model_v2.to(device)

# Calculate class weights for the new training dataset
class_counts_v2 = torch.bincount(
    torch.tensor(train_dataset_v2.targets)
).float()

class_weights_v2 = (
    class_counts_v2.sum() / (2 * class_counts_v2)
).to(device)

# Loss and optimizer
criterion_v2 = nn.CrossEntropyLoss(
    weight=class_weights_v2
)

optimizer_v2 = optim.Adam(
    model_v2.parameters(),
    lr=0.0001
)

print("RESNET-18 V2 CONFIGURED")
print("-" * 40)
print("Device:", device)
print("Architecture: ResNet-18")
print("Output classes:", model_v2.fc.out_features)
print("Training class counts:", class_counts_v2.tolist())
print("Class weights:", class_weights_v2.tolist())
print("Learning rate: 0.0001")
print("Optimizer: Adam")
print("Loss function: Weighted CrossEntropyLoss")

In [ ]:
from pathlib import Path
import torch

EPOCHS = 15

MODEL_V2_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/weights/best_resnet18_v2.pth"
)

MODEL_V2_PATH.parent.mkdir(parents=True, exist_ok=True)

best_val_accuracy_v2 = 0.0
history_v2 = []

for epoch in range(EPOCHS):

    # ---------------- TRAINING ----------------
    model_v2.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_v2:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_v2.zero_grad()

        outputs = model_v2(images)
        loss = criterion_v2(outputs, labels)

        loss.backward()
        optimizer_v2.step()

        train_loss += loss.item() * images.size(0)

        predictions = torch.argmax(outputs, dim=1)
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

    train_loss /= train_total
    train_accuracy = 100 * train_correct / train_total

    # ---------------- VALIDATION ----------------
    model_v2.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader_v2:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_v2(images)
            loss = criterion_v2(outputs, labels)

            val_loss += loss.item() * images.size(0)

            predictions = torch.argmax(outputs, dim=1)
            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_accuracy = 100 * val_correct / val_total

    history_v2.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy
    })

    # Save best model
    if val_accuracy > best_val_accuracy_v2:

        best_val_accuracy_v2 = val_accuracy

        torch.save(
            model_v2.state_dict(),
            MODEL_V2_PATH
        )

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.2f}%"
    )

print("\nV2 TRAINING COMPLETED")
print("Best validation accuracy:", f"{best_val_accuracy_v2:.2f}%")
print("Best model saved at:", MODEL_V2_PATH)

In [ ]:
MODEL_V2_PATH = (
    "/content/drive/MyDrive/WildForest_Fire_Detection/weights/"
    "best_resnet18_v2.pth"
)

model_v2.load_state_dict(
    torch.load(
        MODEL_V2_PATH,
        map_location=device,
        weights_only=True
    )
)

model_v2 = model_v2.to(device)
model_v2.eval()

print("Best ResNet-18 V2 checkpoint loaded!")
print("Device:", device)
print("Evaluation mode:", not model_v2.training)

In [ ]:
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

model_v2.eval()

all_predictions_v2 = []
all_labels_v2 = []
all_probabilities_v2 = []

test_loss_v2 = 0.0
test_total_v2 = 0

with torch.no_grad():

    for images, labels in test_loader_v2:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model_v2(images)
        loss = criterion_v2(outputs, labels)

        probabilities = torch.softmax(outputs, dim=1)
        predictions = torch.argmax(outputs, dim=1)

        test_loss_v2 += loss.item() * images.size(0)
        test_total_v2 += labels.size(0)

        all_predictions_v2.extend(predictions.cpu().numpy())
        all_labels_v2.extend(labels.cpu().numpy())
        all_probabilities_v2.extend(probabilities.cpu().numpy())

test_loss_v2 /= test_total_v2

test_accuracy_v2 = accuracy_score(
    all_labels_v2,
    all_predictions_v2
)

print("RESNET-18 V2 TEST RESULTS")
print("-" * 45)
print("Test images:", test_total_v2)
print(f"Test Loss: {test_loss_v2:.4f}")
print(f"Test Accuracy: {test_accuracy_v2 * 100:.2f}%")

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        all_labels_v2,
        all_predictions_v2,
        target_names=["Fire", "Non_Fire"],
        digits=4
    )
)

print("\nCONFUSION MATRIX")
print(
    confusion_matrix(
        all_labels_v2,
        all_predictions_v2
    )
)

In [ ]:
import json
import pandas as pd
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix

RESULTS_V2_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/results/resnet18_v2"
)

RESULTS_V2_PATH.mkdir(parents=True, exist_ok=True)

# Classification report
report_v2 = classification_report(
    all_labels_v2,
    all_predictions_v2,
    target_names=["Fire", "Non_Fire"],
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report_v2).transpose().to_csv(
    RESULTS_V2_PATH / "classification_report.csv"
)

# Confusion matrix
cm_v2 = confusion_matrix(
    all_labels_v2,
    all_predictions_v2
)

pd.DataFrame(
    cm_v2,
    index=["Actual_Fire", "Actual_Non_Fire"],
    columns=["Predicted_Fire", "Predicted_Non_Fire"]
).to_csv(RESULTS_V2_PATH / "confusion_matrix.csv")

# Metrics
metrics_v2 = {
    "model": "ResNet-18 V2",
    "test_images": len(all_labels_v2),
    "test_accuracy": float(test_accuracy_v2 * 100),
    "test_loss": float(test_loss_v2),
    "false_positives": int(cm_v2[1][0]),
    "false_negatives": int(cm_v2[0][1]),
    "best_validation_accuracy": float(best_val_accuracy_v2)
}

with open(RESULTS_V2_PATH / "test_metrics.json", "w") as f:
    json.dump(metrics_v2, f, indent=4)

print("V2 RESULTS SAVED SUCCESSFULLY")
print("-" * 40)

for file in RESULTS_V2_PATH.iterdir():
    print(file.name)

print("\nResults folder:", RESULTS_V2_PATH)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

YOLO_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/Yolo_dataset/D-Fire"
)

print("Checking dataset folder...")
print("-" * 40)

if YOLO_PATH.exists():

    for item in YOLO_PATH.iterdir():
        print(item.name, "-", "Folder" if item.is_dir() else "File")

else:
    print("Folder not found. Please check its location.")

In [ ]:
import zipfile
from pathlib import Path

ZIP_PATH = (
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "Yolo_dataset/D-Fire/dataset.zip"
)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:

    files = zip_ref.namelist()

    print("ZIP FILE VERIFIED")
    print("-" * 45)

    print("Total files:", len(files))
    print("\nFirst 40 entries:")

    for file in files[:40]:
        print(file)

    print("\nFolder structure keywords:")

    keywords = ["train", "valid", "val", "test", "data.yaml"]

    for keyword in keywords:
        matches = [
            file for file in files
            if keyword.lower() in file.lower()
        ]

        print(f"{keyword}: {len(matches)} matching entries")

In [ ]:
import zipfile

ZIP_PATH = (
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "Yolo_dataset/D-Fire/dataset.zip"
)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:

    print("DATA.YAML CONTENT")
    print("-" * 45)

    yaml_content = zip_ref.read("data.yaml").decode("utf-8")
    print(yaml_content)

    print("\nIMPORTANT DATASET DIRECTORIES")
    print("-" * 45)

    for folder in [
        "data/train/images/",
        "data/train/labels/",
        "data/val/images/",
        "data/val/labels/",
        "data/test/images/",
        "data/test/labels/"
    ]:

        matching_files = [
            name for name in zip_ref.namelist()
            if name.startswith(folder) and not name.endswith("/")
        ]

        print(f"{folder}: {len(matching_files)} files")

In [ ]:
import zipfile
from pathlib import Path

ZIP_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "Yolo_dataset/D-Fire/dataset.zip"
)

EXTRACT_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/extracted"
)

# Safety check
if EXTRACT_PATH.exists():
    raise FileExistsError(
        "Extraction folder already exists. Not overwriting it."
    )

EXTRACT_PATH.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:

    # Check ZIP integrity
    bad_file = zip_ref.testzip()

    if bad_file is not None:
        raise ValueError(f"Corrupted ZIP entry: {bad_file}")

    # Extract dataset
    zip_ref.extractall(EXTRACT_PATH)

print("DATASET EXTRACTED SUCCESSFULLY")
print("-" * 45)
print("Extraction location:", EXTRACT_PATH)

for split in ["train", "val", "test"]:
    image_folder = EXTRACT_PATH / "data" / split / "images"
    label_folder = EXTRACT_PATH / "data" / split / "labels"

    print(f"\n{split.upper()}")
    print("Images:", len(list(image_folder.glob("*"))))
    print("Labels:", len(list(label_folder.glob("*.txt"))))

In [ ]:
from pathlib import Path
from collections import Counter

BASE_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/extracted/data"
)

class_counts = Counter()
invalid_labels = []
empty_labels = []
total_boxes = 0

for split in ["train", "val", "test"]:

    label_folder = BASE_PATH / split / "labels"

    for label_file in label_folder.glob("*.txt"):

        lines = label_file.read_text().strip().splitlines()

        if not lines or lines == [""]:
            empty_labels.append(f"{split}/{label_file.name}")
            continue

        for line in lines:

            values = line.split()

            if len(values) != 5:
                invalid_labels.append(label_file.name)
                continue

            try:
                class_id = int(values[0])
                x, y, w, h = map(float, values[1:])

                if (
                    class_id not in [0, 1]
                    or not all(0 <= v <= 1 for v in [x, y, w, h])
                    or w <= 0
                    or h <= 0
                    or x - w/2 < 0
                    or y - h/2 < 0
                    or x + w/2 > 1
                    or y + h/2 > 1
                ):
                    invalid_labels.append(label_file.name)
                    continue

                class_counts[class_id] += 1
                total_boxes += 1

            except Exception:
                invalid_labels.append(label_file.name)

print("ANNOTATION VALIDATION RESULTS")
print("-" * 45)
print("Total bounding boxes:", total_boxes)
print("Smoke boxes (class 0):", class_counts[0])
print("Fire boxes (class 1):", class_counts[1])
print("Invalid annotation entries:", len(invalid_labels))
print("Empty annotation files:", len(empty_labels))

print("\nFirst invalid files:", invalid_labels[:10])
print("First empty files:", empty_labels[:10])

In [ ]:
from pathlib import Path

BASE_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/extracted/data"
)

checked = 0

for split in ["train", "val", "test"]:

    label_folder = BASE_PATH / split / "labels"

    for label_file in sorted(label_folder.glob("*.txt")):

        lines = label_file.read_text().strip().splitlines()

        for line_number, line in enumerate(lines, start=1):

            values = line.split()
            reason = None

            if len(values) != 5:
                reason = "Incorrect number of values"

            else:
                try:
                    class_id = int(values[0])
                    x, y, w, h = map(float, values[1:])

                    if class_id not in [0, 1]:
                        reason = "Invalid class ID"

                    elif not all(
                        0 <= v <= 1 for v in [x, y, w, h]
                    ):
                        reason = "Coordinate outside 0-1"

                    elif w <= 0 or h <= 0:
                        reason = "Zero or negative box size"

                    elif (
                        x - w/2 < 0 or
                        y - h/2 < 0 or
                        x + w/2 > 1 or
                        y + h/2 > 1
                    ):
                        reason = "Bounding box exceeds image boundary"

                except Exception:
                    reason = "Conversion error"

            if reason:
                print("\nFile:", label_file.name)
                print("Split:", split)
                print("Line:", line_number)
                print("Content:", line)
                print("Problem:", reason)

                checked += 1

                if checked >= 15:
                    break

        if checked >= 15:
            break

    if checked >= 15:
        break

print("\nTotal invalid examples displayed:", checked)

In [ ]:
from pathlib import Path

BASE_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/extracted/data"
)

boundary_issues = []

for split in ["train", "val", "test"]:

    label_folder = BASE_PATH / split / "labels"

    for label_file in label_folder.glob("*.txt"):

        for line_number, line in enumerate(
            label_file.read_text().splitlines(), start=1
        ):

            values = line.split()

            if len(values) != 5:
                continue

            try:
                class_id = int(values[0])
                x, y, w, h = map(float, values[1:])

                if class_id not in [0, 1]:
                    continue

                if not all(
                    0 <= v <= 1 for v in [x, y, w, h]
                ):
                    continue

                if w <= 0 or h <= 0:
                    continue

                left = x - w / 2
                right = x + w / 2
                top = y - h / 2
                bottom = y + h / 2

                if (
                    left < 0 or right > 1
                    or top < 0 or bottom > 1
                ):

                    violation = max(
                        0,
                        -left,
                        right - 1,
                        -top,
                        bottom - 1
                    )

                    boundary_issues.append({
                        "file": label_file.name,
                        "split": split,
                        "violation": violation
                    })

            except Exception:
                continue

print("BOUNDARY VIOLATION ANALYSIS")
print("-" * 40)
print("Total boundary violations:", len(boundary_issues))

if boundary_issues:
    violations = [
        item["violation"] for item in boundary_issues
    ]

    print("Maximum violation:", max(violations))
    print("Average violation:", sum(violations) / len(violations))
    print("Violations up to 0.01:",
          sum(v <= 0.01 for v in violations))
    print("Violations above 0.01:",
          sum(v > 0.01 for v in violations))

In [ ]:
from pathlib import Path

BASE_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/extracted/data"
)

large_violations = []

for split in ["train", "val", "test"]:

    label_folder = BASE_PATH / split / "labels"

    for label_file in label_folder.glob("*.txt"):

        for line_number, line in enumerate(
            label_file.read_text().splitlines(), start=1
        ):

            values = line.split()

            if len(values) != 5:
                continue

            try:
                class_id = int(values[0])
                x, y, w, h = map(float, values[1:])

                left = x - w / 2
                right = x + w / 2
                top = y - h / 2
                bottom = y + h / 2

                violation = max(
                    0,
                    -left,
                    right - 1,
                    -top,
                    bottom - 1
                )

                if violation > 0.01:

                    large_violations.append({
                        "split": split,
                        "file": label_file.name,
                        "line": line_number,
                        "class_id": class_id,
                        "x": x,
                        "y": y,
                        "width": w,
                        "height": h,
                        "violation": violation
                    })

            except Exception:
                continue

print("Large boundary violations:", len(large_violations))
print("-" * 60)

for item in large_violations[:15]:
    print(item)

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

BASE_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/extracted/data"
)

samples = [
    ("train", "AoF04856"),
    ("train", "WEB05093"),
    ("train", "WEB05112"),
    ("train", "WEB05114"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for ax, (split, name) in zip(axes, samples):

    image_folder = BASE_PATH / split / "images"
    label_folder = BASE_PATH / split / "labels"

    image_path = next(
        (p for p in image_folder.glob(name + ".*")),
        None
    )

    label_path = label_folder / f"{name}.txt"

    if image_path is None or not label_path.exists():
        ax.set_title(f"{name}: file not found")
        ax.axis("off")
        continue

    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    img_w, img_h = image.size

    for line in label_path.read_text().splitlines():

        values = line.split()

        if len(values) != 5:
            continue

        class_id = int(values[0])
        x, y, w, h = map(float, values[1:])

        x1 = (x - w / 2) * img_w
        y1 = (y - h / 2) * img_h
        x2 = (x + w / 2) * img_w
        y2 = (y + h / 2) * img_h

        draw.rectangle(
            [x1, y1, x2, y2],
            outline="red" if class_id == 1 else "yellow",
            width=3
        )

        draw.text(
            (max(0, x1), max(0, y1)),
            "Fire" if class_id == 1 else "Smoke",
            fill="white"
        )

    ax.imshow(image)
    ax.set_title(f"{name} | {image.size}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

BASE_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/extracted/data"
)

issues = []

for split in ["train", "val", "test"]:

    label_folder = BASE_PATH / split / "labels"

    for label_file in label_folder.glob("*.txt"):

        for line_number, line in enumerate(
            label_file.read_text().splitlines(), start=1
        ):

            values = line.split()

            if len(values) != 5:
                continue

            try:
                class_id = int(values[0])
                x, y, w, h = map(float, values[1:])

                left = x - w / 2
                right = x + w / 2
                top = y - h / 2
                bottom = y + h / 2

                violation = max(
                    0, -left, right - 1, -top, bottom - 1
                )

                if violation > 0.05:
                    issues.append({
                        "split": split,
                        "file": label_file.name,
                        "line": line_number,
                        "class": class_id,
                        "x": round(x, 4),
                        "y": round(y, 4),
                        "w": round(w, 4),
                        "h": round(h, 4),
                        "violation": round(violation, 4)
                    })

            except Exception:
                continue

issues.sort(
    key=lambda item: item["violation"],
    reverse=True
)

print("Annotations with more than 5% overflow:", len(issues))

for item in issues[:20]:
    print(item)

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

BASE_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/extracted/data"
)

samples = [
    ("train", "WEB09363"),
    ("train", "WEB05162"),
    ("train", "WEB05276"),
    ("val", "WEB05298"),
    ("train", "WEB05362"),
    ("train", "WEB05299"),
]

fig, axes = plt.subplots(3, 2, figsize=(16, 16))
axes = axes.flatten()

for ax, (split, name) in zip(axes, samples):

    image_folder = BASE_PATH / split / "images"
    label_folder = BASE_PATH / split / "labels"

    image_path = next(
        (p for p in image_folder.glob(name + ".*")),
        None
    )

    label_path = label_folder / f"{name}.txt"

    if image_path is None:
        ax.set_title(f"{name}: image not found")
        ax.axis("off")
        continue

    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    img_w, img_h = image.size

    for line in label_path.read_text().splitlines():

        values = line.split()

        if len(values) != 5:
            continue

        class_id = int(values[0])
        x, y, w, h = map(float, values[1:])

        x1 = (x - w / 2) * img_w
        y1 = (y - h / 2) * img_h
        x2 = (x + w / 2) * img_w
        y2 = (y + h / 2) * img_h

        draw.rectangle(
            [x1, y1, x2, y2],
            outline="yellow" if class_id == 0 else "red",
            width=3
        )

        draw.text(
            (max(0, x1), max(0, y1)),
            "Smoke" if class_id == 0 else "Fire",
            fill="white"
        )

    ax.imshow(image)
    ax.set_title(name)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import shutil

SOURCE = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/extracted"
)

DESTINATION = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/cleaned"
)

if DESTINATION.exists():
    print("Cleaned folder already exists.")
    print("Please do not overwrite it without checking its contents.")

else:
    shutil.copytree(SOURCE, DESTINATION)

    print("Dataset copied successfully!")
    print("Original:", SOURCE)
    print("Cleaned:", DESTINATION)

    for split in ["train", "val", "test"]:
        image_count = len(
            list((DESTINATION / "data" / split / "images").glob("*"))
        )
        label_count = len(
            list((DESTINATION / "data" / split / "labels").glob("*.txt"))
        )

        print(
            f"{split}: {image_count} images | "
            f"{label_count} label files"
        )

In [ ]:
from pathlib import Path

CLEANED = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/cleaned"
)

print("CLEANED DATASET INSPECTION")
print("=" * 45)
print("Folder exists:", CLEANED.exists())

for folder in [
    CLEANED / "data" / "train" / "images",
    CLEANED / "data" / "train" / "labels",
    CLEANED / "data" / "val" / "images",
    CLEANED / "data" / "val" / "labels",
    CLEANED / "data" / "test" / "images",
    CLEANED / "data" / "test" / "labels",
]:

    if folder.exists():
        count = len(list(folder.iterdir()))
        print(f"{folder.relative_to(CLEANED)}: {count} files")
    else:
        print(f"MISSING: {folder.relative_to(CLEANED)}")

print("\nYAML files:")

for yaml_file in CLEANED.rglob("*.yaml"):
    print(yaml_file)

In [ ]:
from pathlib import Path

CLEANED = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/cleaned"
)

print("FOLDER STRUCTURE")
print("=" * 50)

for path in sorted(CLEANED.rglob("*")):
    if path.is_dir():
        print("DIR :", path.relative_to(CLEANED))
    else:
        print("FILE:", path.relative_to(CLEANED))

print("\nDATA YAML CONTENT")
print("=" * 50)

yaml_path = CLEANED / "data.yaml"

if yaml_path.exists():
    print(yaml_path.read_text())

In [ ]:
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire"
)

for dataset_name in ["extracted", "cleaned"]:

    print(f"\n{'='*45}")
    print(dataset_name.upper())
    print("="*45)

    for split in ["train", "val", "test"]:

        image_dir = ROOT / dataset_name / "data" / split / "images"
        label_dir = ROOT / dataset_name / "data" / split / "labels"

        if not image_dir.exists() or not label_dir.exists():
            print(f"{split}: FOLDERS MISSING")
            continue

        images = {
            p.stem for p in image_dir.iterdir()
            if p.is_file()
        }

        labels = {
            p.stem for p in label_dir.glob("*.txt")
        }

        print(
            f"{split}: images={len(images)}, "
            f"labels={len(labels)}, "
            f"missing_labels={len(images-labels)}, "
            f"orphan_labels={len(labels-images)}"
        )

In [ ]:
from pathlib import Path

CLEANED = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/cleaned"
)

print("Cleaned folder exists:", CLEANED.exists())

if CLEANED.exists():
    print("\nImmediate contents:")

    for item in sorted(CLEANED.iterdir()):
        if item.is_dir():
            print("FOLDER:", item.name)
        else:
            print("FILE:", item.name)

    print("\nData folder contents:")

    data_folder = CLEANED / "data"

    if data_folder.exists():
        for item in sorted(data_folder.iterdir()):
            print("FOLDER:" if item.is_dir() else "FILE:", item.name)
    else:
        print("No data folder found.")

In [ ]:
from pathlib import Path
import shutil

SOURCE = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/extracted"
)

DESTINATION = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/cleaned_v2"
)

if not SOURCE.exists():
    raise FileNotFoundError("Original extracted dataset not found.")

if DESTINATION.exists():
    print("STOP: cleaned_v2 already exists.")
    print("We will inspect it before making any changes.")

else:
    print("Copying original dataset. Please wait...")

    shutil.copytree(SOURCE, DESTINATION)

    print("\nCopy completed successfully!")

    for split in ["train", "val", "test"]:

        image_dir = DESTINATION / "data" / split / "images"
        label_dir = DESTINATION / "data" / split / "labels"

        image_count = len(list(image_dir.iterdir()))
        label_count = len(list(label_dir.glob("*.txt")))

        print(
            f"{split}: {image_count} images | "
            f"{label_count} labels"
        )

In [ ]:
from pathlib import Path
import math
import csv

ROOT = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/cleaned_v2"
)

DATA_DIR = ROOT / "data"
REPORT_PATH = ROOT / "annotation_repair_report.csv"

repaired_boxes = 0
unchanged_boxes = 0
invalid_lines = []
empty_files = 0

for split in ["train", "val", "test"]:

    label_dir = DATA_DIR / split / "labels"

    for label_file in label_dir.glob("*.txt"):

        original_lines = label_file.read_text().splitlines()
        updated_lines = []

        if not original_lines:
            empty_files += 1
            continue

        for line_number, line in enumerate(original_lines, start=1):

            parts = line.split()

            try:
                if len(parts) != 5:
                    raise ValueError("Expected exactly 5 values")

                cls = int(parts[0])
                x, y, w, h = map(float, parts[1:])

                if cls not in [0, 1]:
                    raise ValueError("Invalid class ID")

                if not all(math.isfinite(v) for v in [x, y, w, h]):
                    raise ValueError("Non-finite coordinate")

                if not (0 <= x <= 1 and 0 <= y <= 1):
                    raise ValueError("Center outside normalized range")

                if not (0 < w <= 1 and 0 < h <= 1):
                    raise ValueError("Invalid width or height")

                # Clip bounding box to image boundaries.
                x1 = max(0.0, x - w / 2)
                y1 = max(0.0, y - h / 2)
                x2 = min(1.0, x + w / 2)
                y2 = min(1.0, y + h / 2)

                if x2 <= x1 or y2 <= y1:
                    raise ValueError("Box has no visible area")

                new_x = (x1 + x2) / 2
                new_y = (y1 + y2) / 2
                new_w = x2 - x1
                new_h = y2 - y1

                if (new_x, new_y, new_w, new_h) != (x, y, w, h):
                    repaired_boxes += 1
                else:
                    unchanged_boxes += 1

                updated_lines.append(
                    f"{cls} {new_x:.8f} {new_y:.8f} "
                    f"{new_w:.8f} {new_h:.8f}"
                )

            except Exception as error:

                invalid_lines.append([
                    split,
                    label_file.name,
                    line_number,
                    line,
                    str(error)
                ])

                # Keep processing other annotations; flag this one for review.

        label_file.write_text(
            "\n".join(updated_lines) +
            ("\n" if updated_lines else "")
        )

print("ANNOTATION REPAIR SUMMARY")
print("-------------------------")
print("Repaired boxes:", repaired_boxes)
print("Unchanged boxes:", unchanged_boxes)
print("Empty label files preserved:", empty_files)
print("Problematic annotation lines:", len(invalid_lines))

with open(REPORT_PATH, "w", newline="", encoding="utf-8") as f:

    writer = csv.writer(f)

    writer.writerow([
        "split",
        "label_file",
        "line_number",
        "original_annotation",
        "reason"
    ])

    writer.writerows(invalid_lines)

print("\nReview report saved to:", REPORT_PATH)

In [ ]:
import pandas as pd
from pathlib import Path

REPORT_PATH = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/cleaned_v2/annotation_repair_report.csv"
)

report = pd.read_csv(REPORT_PATH)

print("Total problematic annotations:", len(report))

display(report)

In [ ]:
from pathlib import Path
import pandas as pd
import math
import csv

ROOT = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/cleaned_v2"
)

REPORT_PATH = ROOT / "annotation_repair_report.csv"

report = pd.read_csv(REPORT_PATH)

recovered = 0
degenerate = []
failed = []

for _, row in report.iterrows():

    split = row["split"]
    label_name = row["label_file"]
    original = row["original_annotation"]

    parts = original.split()

    try:
        cls = int(parts[0])
        x, y, w, h = map(float, parts[1:])

        # Class and coordinate validation
        if cls not in [0, 1]:
            raise ValueError("Invalid class ID")

        if not all(math.isfinite(v) for v in [x, y, w, h]):
            raise ValueError("Non-finite coordinates")

        if not (0 <= x <= 1 and 0 <= y <= 1):
            raise ValueError("Center outside image")

        # Zero-area boxes cannot be recovered safely.
        if w <= 0 or h <= 0:
            degenerate.append([
                split, label_name, original,
                "Zero or negative width/height"
            ])
            continue

        # Clip even if original width/height exceeds 1.
        x1 = max(0.0, x - w / 2)
        y1 = max(0.0, y - h / 2)
        x2 = min(1.0, x + w / 2)
        y2 = min(1.0, y + h / 2)

        if x2 <= x1 or y2 <= y1:
            raise ValueError("No visible box area after clipping")

        new_x = (x1 + x2) / 2
        new_y = (y1 + y2) / 2
        new_w = x2 - x1
        new_h = y2 - y1

        repaired_line = (
            f"{cls} {new_x:.8f} {new_y:.8f} "
            f"{new_w:.8f} {new_h:.8f}"
        )

        label_path = (
            ROOT / "data" / split / "labels" / label_name
        )

        with open(label_path, "a", encoding="utf-8") as f:
            f.write(repaired_line + "\n")

        recovered += 1

    except Exception as error:
        failed.append([
            split, label_name, original, str(error)
        ])

# Save zero-area annotations separately.
degenerate_path = ROOT / "degenerate_annotations.csv"

with open(degenerate_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "split", "label_file", "original_annotation", "reason"
    ])
    writer.writerows(degenerate)

# Save any unexpected failures separately.
failed_path = ROOT / "unrecovered_annotations.csv"

with open(failed_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "split", "label_file", "original_annotation", "reason"
    ])
    writer.writerows(failed)

print("RECOVERY SUMMARY")
print("----------------")
print("Oversized annotations recovered:", recovered)
print("Zero-area annotations excluded:", len(degenerate))
print("Unexpected failures:", len(failed))
print("\nDegenerate report:", degenerate_path)
print("Failure report:", failed_path)

In [ ]:
from pathlib import Path
from PIL import Image
import math

ROOT = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/cleaned_v2"
)

# Correct the path if needed:
ROOT = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/cleaned_v2"
)

total_images = 0
total_labels = 0
total_boxes = 0
empty_labels = 0
invalid_annotations = []
missing_labels = []
orphan_labels = []
unreadable_images = []

class_counts = {0: 0, 1: 0}

for split in ["train", "val", "test"]:

    image_dir = ROOT / "data" / split / "images"
    label_dir = ROOT / "data" / split / "labels"

    images = {
        p.stem: p for p in image_dir.iterdir()
        if p.is_file()
    }

    labels = {
        p.stem: p for p in label_dir.glob("*.txt")
    }

    total_images += len(images)
    total_labels += len(labels)

    missing_labels.extend(
        [(split, name) for name in images.keys() - labels.keys()]
    )

    orphan_labels.extend(
        [(split, name) for name in labels.keys() - images.keys()]
    )

    for image_path in images.values():
        try:
            with Image.open(image_path) as img:
                img.verify()
        except Exception as error:
            unreadable_images.append(
                (split, image_path.name, str(error))
            )

    for label_path in labels.values():

        lines = label_path.read_text().splitlines()

        if not lines:
            empty_labels += 1
            continue

        for line_number, line in enumerate(lines, start=1):

            parts = line.split()

            try:
                if len(parts) != 5:
                    raise ValueError("Expected five values")

                cls = int(parts[0])
                x, y, w, h = map(float, parts[1:])

                if cls not in [0, 1]:
                    raise ValueError("Invalid class ID")

                if not all(math.isfinite(v) for v in [x, y, w, h]):
                    raise ValueError("Non-finite coordinates")

                if not (
                    0 <= x <= 1 and
                    0 <= y <= 1 and
                    0 < w <= 1 and
                    0 < h <= 1
                ):
                    raise ValueError("Coordinates outside valid range")

                if x - w / 2 < -1e-6 or x + w / 2 > 1 + 1e-6:
                    raise ValueError("Horizontal boundary violation")

                if y - h / 2 < -1e-6 or y + h / 2 > 1 + 1e-6:
                    raise ValueError("Vertical boundary violation")

                total_boxes += 1
                class_counts[cls] += 1

            except Exception as error:
                invalid_annotations.append(
                    (split, label_path.name, line_number, str(error))
                )

print("FINAL DATASET VALIDATION")
print("=========================")
print("Total images:", total_images)
print("Total label files:", total_labels)
print("Total valid bounding boxes:", total_boxes)
print("Empty label files:", empty_labels)

print("\nClass distribution:")
print("Smoke (0):", class_counts[0])
print("Fire (1):", class_counts[1])

print("\nProblems:")
print("Invalid annotations:", len(invalid_annotations))
print("Missing labels:", len(missing_labels))
print("Orphan labels:", len(orphan_labels))
print("Unreadable images:", len(unreadable_images))

if invalid_annotations:
    print("\nFirst invalid annotations:")
    print(invalid_annotations[:10])

if missing_labels:
    print("\nFirst missing labels:")
    print(missing_labels[:10])

if orphan_labels:
    print("\nFirst orphan labels:")
    print(orphan_labels[:10])

if unreadable_images:
    print("\nFirst unreadable images:")
    print(unreadable_images[:10])

In [ ]:
import random
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

ROOT = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/cleaned_v2"
)

IMAGE_DIR = ROOT / "data/train/images"
LABEL_DIR = ROOT / "data/train/labels"

CLASS_NAMES = {
    0: "Smoke",
    1: "Fire"
}

image_files = list(IMAGE_DIR.iterdir())

# Select 6 random images that contain annotations.
samples = []

random.shuffle(image_files)

for image_path in image_files:

    label_path = LABEL_DIR / f"{image_path.stem}.txt"

    if label_path.exists() and label_path.stat().st_size > 0:
        samples.append(image_path)

    if len(samples) == 6:
        break

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for ax, image_path in zip(axes, samples):

    image = Image.open(image_path).convert("RGB")
    width, height = image.size

    ax.imshow(image)

    label_path = LABEL_DIR / f"{image_path.stem}.txt"

    for line in label_path.read_text().splitlines():

        cls, x, y, w, h = map(float, line.split())

        x1 = (x - w / 2) * width
        y1 = (y - h / 2) * height

        box_width = w * width
        box_height = h * height

        rect = patches.Rectangle(
            (x1, y1),
            box_width,
            box_height,
            linewidth=2,
            edgecolor="lime" if int(cls) == 0 else "red",
            facecolor="none"
        )

        ax.add_patch(rect)

        ax.text(
            x1,
            max(0, y1 - 5),
            CLASS_NAMES[int(cls)],
            color="white",
            fontsize=10,
            bbox=dict(facecolor="black", alpha=0.7)
        )

    ax.set_title(image_path.name)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import yaml

ROOT = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/cleaned_v2"
)

data_config = {
    "path": str(ROOT),
    "train": "data/train/images",
    "val": "data/val/images",
    "test": "data/test/images",
    "names": {
        0: "smoke",
        1: "fire"
    },
    "nc": 2
}

yaml_path = ROOT / "data.yaml"

with open(yaml_path, "w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print("YOLO configuration created successfully!")
print("\nConfiguration:\n")
print(yaml_path.read_text())

print("File exists:", yaml_path.exists())

In [ ]:
import torch
import platform

print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU name:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

    total_memory = torch.cuda.get_device_properties(0).total_memory
    print("GPU memory:", round(total_memory / (1024**3), 2), "GB")

else:
    print("\nNo GPU detected.")
    print("In Colab, select Runtime > Change runtime type > T4 GPU.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil

SOURCE = Path(
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/cleaned_v2"
)

if not SOURCE.exists():
    raise FileNotFoundError(f"Folder not found: {SOURCE}")

OUTPUT = SOURCE.parent / "cleaned_v2_kaggle.zip"

if OUTPUT.exists():
    print("ZIP already exists:", OUTPUT)
else:
    print("Creating ZIP. Please wait...")
    shutil.make_archive(
        str(OUTPUT.with_suffix("")),
        "zip",
        root_dir=SOURCE.parent,
        base_dir=SOURCE.name
    )

print("\nZIP created:", OUTPUT)
print("Size (GB):", round(OUTPUT.stat().st_size / (1024**3), 2))

In [ ]:
from google.colab import files

ZIP_PATH = (
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/cleaned_v2_kaggle.zip"
)

files.download(ZIP_PATH)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PROJECT_PATH = "/content/drive/MyDrive/WildForest_Fire_Detection"

print("Project folder exists:", os.path.exists(PROJECT_PATH))

if os.path.exists(PROJECT_PATH):
    print("\nProject contents:")
    for item in os.listdir(PROJECT_PATH):
        print("-", item)
else:
    print("Project folder not found. Check the Google Drive folder name.")

In [ ]:
import os
from collections import Counter

PROJECT_PATH = "/content/drive/MyDrive/WildForest_Fire_Detection"

for folder in os.listdir(PROJECT_PATH):
    folder_path = os.path.join(PROJECT_PATH, folder)

    if os.path.isdir(folder_path):
        file_count = 0
        folder_count = 0
        extensions = Counter()

        for root, dirs, files in os.walk(folder_path):
            folder_count += len(dirs)

            for file in files:
                file_count += 1
                ext = os.path.splitext(file)[1].lower()
                extensions[ext if ext else "[no extension]"] += 1

        print("\n" + "=" * 55)
        print("FOLDER:", folder)
        print("Subfolders:", folder_count)
        print("Total files:", file_count)
        print("File types:", dict(extensions))

In [ ]:
import os
from collections import Counter

PROJECT_PATH = "/content/drive/MyDrive/WildForest_Fire_Detection"

print("PROJECT FILE INVENTORY")
print("=" * 65)

total_size = 0
important_files = []

for root, dirs, files in os.walk(PROJECT_PATH):

    # Skip unnecessary hidden/system folders
    dirs[:] = [d for d in dirs if not d.startswith(".")]

    for file in files:
        path = os.path.join(root, file)
        size = os.path.getsize(path)
        total_size += size

        ext = os.path.splitext(file)[1].lower()

        if ext in [".ipynb", ".py", ".md", ".yaml", ".yml",
                   ".txt", ".csv", ".json", ".pth"]:
            relative_path = os.path.relpath(path, PROJECT_PATH)

            important_files.append(
                (relative_path, round(size / (1024 * 1024), 2))
            )

print("\nIMPORTANT PROJECT FILES:")

for path, size in important_files:
    print(f"{path}  |  {size} MB")

print("\n" + "=" * 65)
print("TOTAL PROJECT SIZE:", round(total_size / (1024**3), 2), "GB")
print("IMPORTANT FILE COUNT:", len(important_files))

In [ ]:
import os

PROJECT_PATH = "/content/drive/MyDrive/WildForest_Fire_Detection"

allowed_extensions = {
    ".ipynb", ".py", ".md", ".yaml", ".yml",
    ".json", ".csv", ".pth", ".pdf"
}

print("PROJECT CODE AND DOCUMENTATION FILES")
print("=" * 65)

for root, dirs, files in os.walk(PROJECT_PATH):

    # Skip dataset directories
    dirs[:] = [
        d for d in dirs
        if d.lower() not in ["dataset", "yolo_dataset", "yolo_dataset"]
    ]

    for file in files:
        ext = os.path.splitext(file)[1].lower()

        if ext in allowed_extensions:
            path = os.path.join(root, file)
            size_mb = os.path.getsize(path) / (1024 * 1024)

            print(f"{os.path.relpath(path, PROJECT_PATH)}  |  {size_mb:.2f} MB")

In [ ]:
import os

PROJECT_PATH = "/content/drive/MyDrive/WildForest_Fire_Detection"

print("PROJECT ROOT CONTENTS")
print("=" * 60)

for item in sorted(os.listdir(PROJECT_PATH)):
    path = os.path.join(PROJECT_PATH, item)

    if os.path.isdir(path):
        print(f"[FOLDER] {item}")
    else:
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"[FILE]   {item} ({size_mb:.2f} MB)")

In [ ]:
import os

REPO_URL = "https://github.com/Aman-singh0513/BT013---AI-Based-Forest-Fire-Detection-Using-Satellite-Imagery.git"
REPO_PATH = "/content/BT013-Forest-Fire-Detection"

if not os.path.exists(REPO_PATH):
    !git clone "{REPO_URL}" "{REPO_PATH}"
else:
    print("Repository already cloned.")

print("\nEXISTING GITHUB REPOSITORY CONTENTS")
print("=" * 60)

for item in sorted(os.listdir(REPO_PATH)):
    path = os.path.join(REPO_PATH, item)
    print("[FOLDER]" if os.path.isdir(path) else "[FILE]  ", item)

In [ ]:
import os
import shutil

PROJECT_PATH = "/content/drive/MyDrive/WildForest_Fire_Detection"
REPO_PATH = "/content/BT013-Forest-Fire-Detection"

# Copy trained model weights
source_weights = os.path.join(PROJECT_PATH, "weights")
target_weights = os.path.join(REPO_PATH, "weights")

os.makedirs(target_weights, exist_ok=True)

for file in ["best_resnet18.pth", "best_resnet18_v2.pth"]:
    shutil.copy2(
        os.path.join(source_weights, file),
        os.path.join(target_weights, file)
    )

# Copy evaluation results
source_results = os.path.join(PROJECT_PATH, "results")
target_results = os.path.join(REPO_PATH, "results")

shutil.copytree(source_results, target_results, dirs_exist_ok=True)

print("FILES COPIED SUCCESSFULLY")
print("=" * 50)

for root, dirs, files in os.walk(REPO_PATH):
    if ".git" in root.split(os.sep):
        continue

    for file in files:
        path = os.path.join(root, file)
        print(os.path.relpath(path, REPO_PATH),
              f"| {os.path.getsize(path) / (1024 * 1024):.2f} MB")

In [ ]:
README_PATH = "/content/BT013-Forest-Fire-Detection/README.md"

readme_content = r"""
# AI-Based Forest Fire Detection Using Satellite Imagery

## Project Overview

This project explores deep learning techniques for forest fire detection
using satellite imagery. It currently includes a ResNet-18 image
classification stage and a YOLOv8 fire and smoke object-detection stage.

## Project Structure

```text
BT013-Forest-Fire-Detection/
├── weights/
│   ├── best_resnet18.pth
│   └── best_resnet18_v2.pth
├── results/
│   ├── resnet18_baseline/
│   └── resnet18_v2/
└── README.md

In [ ]:
from pathlib import Path

repo_path = Path("/content/BT013-Forest-Fire-Detection")
readme_path = repo_path / "README.md"

readme_content = [
    "# AI-Based Forest Fire Detection Using Satellite Imagery",
    "",
    "## Project Overview",
    "",
    "This project explores deep learning techniques for forest fire detection using satellite imagery.",
    "It includes a ResNet-18 image classification stage and preparation for YOLOv8 fire and smoke detection.",
    "",
    "## Project Structure",
    "",
    "```text",
    "BT013-Forest-Fire-Detection/",
    "├── weights/",
    "│   ├── best_resnet18.pth",
    "│   └── best_resnet18_v2.pth",
    "├── results/",
    "│   ├── resnet18_baseline/",
    "│   └── resnet18_v2/",
    "└── README.md",
    "```",
    "",
    "## Stage 1: ResNet-18 Image Classification",
    "",
    "### Baseline Model",
    "- Architecture: ResNet-18",
    "- Input size: 224 x 224",
    "- Optimizer: Adam",
    "- Learning rate: 0.0001",
    "- Loss: Weighted CrossEntropyLoss",
    "- Training: 15 epochs",
    "- Test accuracy: 98% (49/50)",
    "- Checkpoint: `weights/best_resnet18.pth`",
    "",
    "### Improved Model (V2)",
    "- Added 92 manually approved hard-negative images to the non-fire training class.",
    "- Training: 15 epochs",
    "- Best validation accuracy: 99.57% (epoch 13)",
    "- Test accuracy: 100% (50/50)",
    "- Test loss: 0.0122",
    "- Checkpoint: `weights/best_resnet18_v2.pth`",
    "",
    "**Evaluation note:** Test results are based on 50 images and do not guarantee real-world performance.",
    "",
    "## Stage 2: YOLOv8 Fire and Smoke Detection",
    "",
    "Dataset: D-Fire",
    "",
    "| Split | Images |",
    "|---|---:|",
    "| Train | 14,122 |",
    "| Validation | 3,099 |",
    "| Test | 4,306 |",
    "| Total | 21,527 |",
    "",
    "Class IDs:",
    "- `0`: smoke",
    "- `1`: fire",
    "",
    "### Annotation Validation",
    "- Valid bounding boxes: 26,539",
    "- Smoke boxes: 11,854",
    "- Fire boxes: 14,685",
    "- Empty label files preserved: 9,838",
    "- Invalid annotations after repair: 0",
    "- Missing labels: 0",
    "- Orphan labels: 0",
    "- Unreadable images: 0",
    "",
    "**Current status:** Dataset preparation and structural validation are complete. YOLOv8 training has not started.",
    "",
    "## Results",
    "",
    "Evaluation files are stored in:",
    "- `results/resnet18_baseline/`",
    "- `results/resnet18_v2/`",
    "",
    "These include test metrics, classification reports, confusion matrices, and other evaluation outputs.",
    "",
    "## Dataset Availability",
    "",
    "The full datasets are not included in this repository because of their large size.",
    "They are stored separately in Google Drive. The YOLOv8 configuration must point to the prepared `cleaned_v2` dataset before training.",
    "",
    "## Development Status",
    "- [x] ResNet-18 baseline training and evaluation",
    "- [x] Hard-negative dataset improvement",
    "- [x] ResNet-18 V2 training and evaluation",
    "- [x] D-Fire dataset preparation",
    "- [x] YOLO annotation structural validation",
    "- [ ] YOLOv8 model training",
    "- [ ] YOLOv8 evaluation",
    "- [ ] Real-time detection integration",
    "",
    "## Important",
    "",
    "The ResNet-18 checkpoints are PyTorch model weights. Training source code and complete datasets are not currently included in this repository.",
]

readme_path.write_text("\n".join(readme_content) + "\n", encoding="utf-8")

print("README.md created successfully!")
print("File size:", readme_path.stat().st_size, "bytes")
print("\nREADME preview:\n")
print(readme_path.read_text(encoding="utf-8")[:1200])

In [ ]:
import subprocess
from pathlib import Path

REPO_PATH = Path("/content/BT013-Forest-Fire-Detection")

# Check Git status
result = subprocess.run(
    ["git", "status", "--short", "--untracked-files=all"],
    cwd=REPO_PATH,
    capture_output=True,
    text=True
)

print("FILES GIT IS PREPARING")
print("=" * 60)
print(result.stdout)

# Check total repository size, excluding Git metadata
total_size = sum(
    p.stat().st_size
    for p in REPO_PATH.rglob("*")
    if p.is_file() and ".git" not in p.parts
)

print("=" * 60)
print(f"Total project upload size: {total_size / (1024 * 1024):.2f} MB")

In [ ]:
import subprocess

REPO_PATH = "/content/BT013-Forest-Fire-Detection"

# Stage README, model weights, and evaluation results
subprocess.run(
    ["git", "add", "README.md", "weights/", "results/"],
    cwd=REPO_PATH,
    check=True
)

# Review staged files
result = subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO_PATH,
    capture_output=True,
    text=True,
    check=True
)

print("STAGED FILES FOR COMMIT")
print("=" * 60)
print(result.stdout)

# Show staged file count
count = subprocess.run(
    ["git", "diff", "--cached", "--name-only"],
    cwd=REPO_PATH,
    capture_output=True,
    text=True,
    check=True
)

print("Total staged files:", len(count.stdout.strip().splitlines()))

In [ ]:
import subprocess

REPO_PATH = "/content/BT013-Forest-Fire-Detection"

commit_message = "Add ResNet18 models and evaluation results"

result = subprocess.run(
    ["git", "commit", "-m", commit_message],
    cwd=REPO_PATH,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print("ERROR:")
    print(result.stderr)
else:
    print("\nCommit created successfully!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_PATH = "/content/drive/MyDrive/WildForest_Fire_Detection"

ZIP_PATH = os.path.join(
    PROJECT_PATH,
    "YOLO_Dataset",
    "D-Fire",
    "cleaned_v2_kaggle.zip"
)

print("\nDATASET ZIP VERIFICATION")
print("=" * 50)

if os.path.exists(ZIP_PATH):
    size_gb = os.path.getsize(ZIP_PATH) / (1024 ** 3)
    print("Status: FOUND")
    print(f"File size: {size_gb:.2f} GB")
else:
    print("Status: NOT FOUND")
    print("Expected:", ZIP_PATH)

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU unavailable. Current runtime is CPU-only.")

In [ ]:
import os
import yaml

DATASET_PATH = (
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/cleaned_v2"
)

YAML_PATH = os.path.join(DATASET_PATH, "data.yaml")

print("YOLO DATASET VERIFICATION")
print("=" * 55)

if not os.path.isdir(DATASET_PATH):
    print("ERROR: cleaned_v2 folder not found.")
else:
    print("Dataset folder: FOUND")

    for split in ["train", "val", "test"]:
        image_dir = os.path.join(DATASET_PATH, "data", split, "images")
        label_dir = os.path.join(DATASET_PATH, "data", split, "labels")

        images = len(os.listdir(image_dir)) if os.path.isdir(image_dir) else 0
        labels = len(os.listdir(label_dir)) if os.path.isdir(label_dir) else 0

        print(f"{split.upper():5} | Images: {images:,} | Labels: {labels:,}")

    print("\nYAML CONFIGURATION")
    print("-" * 55)

    if os.path.isfile(YAML_PATH):
        with open(YAML_PATH, "r") as f:
            config = yaml.safe_load(f)

        print(yaml.dump(config, sort_keys=False))
    else:
        print("ERROR: data.yaml not found.")

In [ ]:
%pip install ultralytics

In [ ]:
import torch
import ultralytics
from ultralytics import YOLO

print("ENVIRONMENT CHECK")
print("=" * 50)

print("Ultralytics version:", ultralytics.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2),
          "GB")

print("\nYOLO IMPORT: SUCCESS")

In [ ]:
import os
import shutil

PROJECT_PATH = "/content/drive/MyDrive/WildForest_Fire_Detection"

RUNS_PATH = os.path.join(
    PROJECT_PATH,
    "YOLO_Training_Results"
)

os.makedirs(RUNS_PATH, exist_ok=True)

total, used, free = shutil.disk_usage("/content")

print("TRAINING STORAGE CHECK")
print("=" * 50)

print(f"Colab storage free: {free / (1024**3):.2f} GB")
print("Drive output directory:", RUNS_PATH)
print("Output directory exists:", os.path.isdir(RUNS_PATH))

print("\nDATASET:")
print("cleaned_v2")

print("\nMODEL:")
print("YOLOv8n pretrained")

In [ ]:
import os
import yaml

DATASET_PATH = (
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/cleaned_v2"
)

DATA_YAML = os.path.join(DATASET_PATH, "data.yaml")

with open(DATA_YAML, "r") as f:
    config = yaml.safe_load(f)

print("YOLO CONFIGURATION CHECK")
print("=" * 55)

for split in ["train", "val", "test"]:
    image_path = os.path.join(DATASET_PATH, config[split])

    print(f"\n{split.upper()} PATH:")
    print(image_path)
    print("Exists:", os.path.isdir(image_path))

    if os.path.isdir(image_path):
        images = [
            f for f in os.listdir(image_path)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]
        print("Image count:", len(images))

print("\nCLASS CONFIGURATION:")
print(config["names"])
print("Number of classes:", config["nc"])

In [ ]:
import os

SOURCE_PATH = (
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/cleaned_v2"
)

total_bytes = 0
file_count = 0

for root, dirs, files in os.walk(SOURCE_PATH):
    for file in files:
        path = os.path.join(root, file)
        total_bytes += os.path.getsize(path)
        file_count += 1

print("DATASET STORAGE REPORT")
print("=" * 50)
print(f"Total files: {file_count:,}")
print(f"Dataset size: {total_bytes / (1024**3):.2f} GB")
print(f"Colab free storage: 112.45 GB (previous check)")

In [ ]:
import os
import shutil
import time

SOURCE_PATH = (
    "/content/drive/MyDrive/WildForest_Fire_Detection/"
    "YOLO_Dataset/D-Fire/cleaned_v2"
)

DESTINATION_PATH = "/content/cleaned_v2"

# Prevent accidental overwriting
if os.path.exists(DESTINATION_PATH):
    print("Destination already exists.")
    print("Please tell me before proceeding.")
else:
    print("Starting dataset copy...")
    start_time = time.time()

    shutil.copytree(SOURCE_PATH, DESTINATION_PATH)

    elapsed = time.time() - start_time

    print("\nDATASET COPY COMPLETED")
    print("=" * 45)
    print(f"Destination: {DESTINATION_PATH}")
    print(f"Time taken: {elapsed / 60:.2f} minutes")

In [ ]:
import os

BASE_PATH = "/content/cleaned_v2"

splits = ["train", "val", "test"]

print("COPIED DATASET VERIFICATION")
print("=" * 55)

for split in splits:
    image_dir = os.path.join(BASE_PATH, "data", split, "images")
    label_dir = os.path.join(BASE_PATH, "data", split, "labels")

    images = [
        f for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    labels = [
        f for f in os.listdir(label_dir)
        if f.lower().endswith(".txt")
    ]

    print(f"\n{split.upper()}")
    print(f"Images : {len(images):,}")
    print(f"Labels : {len(labels):,}")
    print(f"Match  : {len(images) == len(labels)}")

print("\nVerification completed.")

In [ ]:
import yaml
import os

DATASET_PATH = "/content/cleaned_v2"

data_config = {
    "path": DATASET_PATH,
    "train": "data/train/images",
    "val": "data/val/images",
    "test": "data/test/images",
    "names": {
        0: "smoke",
        1: "fire"
    },
    "nc": 2
}

YAML_PATH = os.path.join(DATASET_PATH, "data.yaml")

with open(YAML_PATH, "w") as file:
    yaml.safe_dump(data_config, file, sort_keys=False)

print("YOLO CONFIGURATION CREATED")
print("=" * 45)

with open(YAML_PATH, "r") as file:
    print(file.read())

print("File exists:", os.path.exists(YAML_PATH))

In [ ]:
from ultralytics import YOLO
import torch

print("Loading pretrained YOLOv8 Nano...")

model = YOLO("yolov8n.pt")

print("\nMODEL LOADED SUCCESSFULLY")
print("=" * 45)

print("Model:", model.ckpt_path)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

print("\nModel information:")
model.info()

In [ ]:
from ultralytics import YOLO
import torch
import os

# Load pretrained model
model = YOLO("yolov8n.pt")

# Dataset configuration
DATA_YAML = "/content/cleaned_v2/data.yaml"

# Google Drive output directory
RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "WildForest_Fire_Detection/YOLO_Training_Results"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Starting YOLOv8 Fire and Smoke Detection Training")
print("=" * 55)
print("GPU:", torch.cuda.get_device_name(0))
print("Dataset:", DATA_YAML)
print("Output:", RESULTS_DIR)

results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=10,
    save=True,
    save_period=10,
    project=RESULTS_DIR,
    name="yolov8n_dfire_v1",
    exist_ok=False,
    pretrained=True,
    seed=42,
    deterministic=True,
    plots=True,
    verbose=True
)

print("\nTRAINING PROCESS COMPLETED")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "WildForest_Fire_Detection/YOLO_Training_Results/"
    "yolov8n_dfire_v1"
)

print("Checking saved training results...")

if os.path.exists(RESULTS_DIR):
    for item in sorted(os.listdir(RESULTS_DIR)):
        print(item)
else:
    print("ERROR: Training results folder not found.")

In [ ]:
import os
import pandas as pd

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "WildForest_Fire_Detection/YOLO_Training_Results/"
    "yolov8n_dfire_v1"
)

# Check model checkpoints
WEIGHTS_DIR = os.path.join(RESULTS_DIR, "weights")

print("SAVED MODEL CHECKPOINTS")
print("=" * 45)

for file in os.listdir(WEIGHTS_DIR):
    path = os.path.join(WEIGHTS_DIR, file)
    print(f"{file} — {os.path.getsize(path) / (1024 * 1024):.2f} MB")

# Inspect training history
df = pd.read_csv(os.path.join(RESULTS_DIR, "results.csv"))
df.columns = df.columns.str.strip()

metric = "metrics/mAP50-95(B)"

if metric in df.columns:
    best_index = df[metric].idxmax()

    print("\nBEST VALIDATION EPOCH")
    print("=" * 45)
    print("Epoch:", best_index + 1)
    print("mAP50-95:", round(df.loc[best_index, metric], 4))
    print("mAP50:", round(df.loc[best_index, "metrics/mAP50(B)"], 4))

In [ ]:
import os
from IPython.display import display, Image

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "WildForest_Fire_Detection/YOLO_Training_Results/"
    "yolov8n_dfire_v1"
)

graphs = [
    "results.png",
    "confusion_matrix_normalized.png",
    "BoxPR_curve.png",
    "BoxF1_curve.png"
]

for filename in graphs:
    path = os.path.join(RESULTS_DIR, filename)

    print(f"\n{'=' * 55}")
    print(filename)
    print("=" * 55)

    if os.path.exists(path):
        display(Image(filename=path, width=850))
    else:
        print("File not found:", filename)

In [ ]:
import os
from IPython.display import display, Image

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "WildForest_Fire_Detection/YOLO_Training_Results/"
    "yolov8n_dfire_v1"
)

matrix_path = os.path.join(
    RESULTS_DIR,
    "confusion_matrix.png"
)

print("ORIGINAL CONFUSION MATRIX")
print("=" * 50)

display(Image(filename=matrix_path, width=850))

In [ ]:
%pip install ultralytics==8.4.215

In [ ]:
import os

PROJECT_DIR = (
    "/content/drive/MyDrive/"
    "WildForest_Fire_Detection"
)

print("PROJECT ROOT")
print("=" * 60)

for item in sorted(os.listdir(PROJECT_DIR)):
    path = os.path.join(PROJECT_DIR, item)

    if os.path.isdir(path):
        print(f"[FOLDER] {item}")
    else:
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"[FILE]   {item} ({size_mb:.2f} MB)")

print("\nPOSSIBLE NOTEBOOKS")
print("=" * 60)

for root, dirs, files in os.walk(PROJECT_DIR):
    dirs[:] = [
        d for d in dirs
        if d not in ["train", "val", "test", "images", "labels"]
    ]

    for filename in files:
        if filename.endswith(".ipynb"):
            print(os.path.join(root, filename))

In [ ]:
import os

PROJECT_DIR = (
    "/content/drive/MyDrive/"
    "WildForest_Fire_Detection"
)

SKIP_DIRS = {
    "images", "labels", "train", "val", "test",
    ".git", "__pycache__"
}

print("SOURCE CODE AND CONFIGURATION FILES")
print("=" * 65)

extensions = (
    ".py", ".ipynb", ".yaml", ".yml",
    ".md", ".txt", ".json"
)

for root, dirs, files in os.walk(PROJECT_DIR):
    dirs[:] = [d for d in dirs if d not in SKIP_DIRS]

    for filename in sorted(files):
        if filename.endswith(extensions):
            path = os.path.join(root, filename)
            relative = os.path.relpath(path, PROJECT_DIR)
            print(relative)

print("\nMODEL CHECKPOINTS")
print("=" * 65)

for folder in ["weights", "YOLO_Training_Results"]:
    folder_path = os.path.join(PROJECT_DIR, folder)

    if os.path.exists(folder_path):
        for root, dirs, files in os.walk(folder_path):
            dirs[:] = [d for d in dirs if d not in SKIP_DIRS]

            for filename in sorted(files):
                if filename.endswith(".pt"):
                    path = os.path.join(root, filename)
                    size = os.path.getsize(path) / (1024 * 1024)
                    print(f"{os.path.relpath(path, PROJECT_DIR)} — {size:.2f} MB")